In [13]:
# HEALTH DATA INTEGRATION


import pandas as pd
import numpy as np
import os

print("=== Phase 5: FINAL CLEAN DATASET ===\n")

DATA_PROCESSED = r"I:\Data Science Projects\disease_prediction\data\processed"

# LOAD DATA
df = pd.read_csv(
    os.path.join(DATA_PROCESSED, "district_monthly_climate_features_2010_2024.csv")
)

print("Raw shape:", df.shape)

assert "district" in df.columns, "❌ district column missing!"

# PREPROCESS
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(["district", "date"]).reset_index(drop=True)

df["month"] = df["date"].dt.month
df["year"] = df["date"].dt.year

df["is_rainy_season"] = df["month"].isin([3,4,5,9,10,11]).astype(int)

# Seasonality
df["sin_month"] = np.sin(2 * np.pi * df["month"] / 12)
df["cos_month"] = np.cos(2 * np.pi * df["month"] / 12)

# LAG FEATURES
df["tp_lag1"] = df.groupby("district")["tp"].shift(1)
df["tp_lag2"] = df.groupby("district")["tp"].shift(2)
df["temp_lag1"] = df.groupby("district")["t2m"].shift(1)

lag_cols = ["tp_lag1", "tp_lag2", "temp_lag1"]

df[lag_cols] = df.groupby("district")[lag_cols].transform(
    lambda x: x.bfill().ffill()
)

# TREND
df["year_trend"] = (df["year"] - df["year"].min()) * 0.02

# RISK MODEL
np.random.seed(42)

base = 0.2

tp_norm = (df["tp"] - df["tp"].mean()) / df["tp"].std()
t2m_norm = (df["t2m"] - df["t2m"].mean()) / df["t2m"].std()

df["vector_risk"] = (
    base
    + 0.25 * t2m_norm
    + 0.35 * tp_norm
    + 0.25 * df["tp_lag1"]
    + 0.15 * df["tp_lag2"]
    + 0.15 * df["is_rainy_season"]
    + df["year_trend"]
)

df["water_risk"] = (
    base
    + 0.50 * tp_norm
    + 0.30 * df["tp_lag1"]
    + 0.20 * df["tp_lag2"]
    + 0.10 * df["is_rainy_season"]
    + df["year_trend"]
)

# Noise
df["noise"] = np.random.normal(0, 0.05, len(df))

df["vector_risk"] += df["noise"]
df["water_risk"] += df["noise"]

df["vector_risk"] = df["vector_risk"].clip(0, 2)
df["water_risk"] = df["water_risk"].clip(0, 2)

# CASE GENERATION
df["season_factor"] = np.where(df["is_rainy_season"], 1.4, 1.0)

df["vector_borne_cases"] = np.random.poisson(
    40 * df["vector_risk"] * df["season_factor"]
)

df["waterborne_cases"] = np.random.poisson(
    15 * df["water_risk"] * df["season_factor"]
)

# SPATIAL
df["spatial_lag_cases"] = df.groupby("date")["vector_borne_cases"].shift(1)
df["spatial_lag_cases"] = df["spatial_lag_cases"].fillna(
    df.groupby("date")["vector_borne_cases"].transform("mean")
)

df["vector_borne_cases"] += (0.1 * df["spatial_lag_cases"]).astype(int)

# TOTAL
df["total_cases"] = df["vector_borne_cases"] + df["waterborne_cases"]

# POPULATION
district_pop = {
    d: np.random.randint(50000, 500000)
    for d in df["district"].unique()
}

df["population"] = df["district"].map(district_pop)
df["incidence_rate"] = (df["total_cases"] / df["population"]) * 100000

# OUTBREAK (FIXED)
df["rolling_mean"] = df.groupby("district")["total_cases"].transform(
    lambda x: x.rolling(6, min_periods=1).mean()
)

df["rolling_std"] = df.groupby("district")["total_cases"].transform(
    lambda x: x.rolling(6, min_periods=1).std().fillna(0)
)

# THRESHOLD
df["outbreak"] = (
    df["total_cases"] > (df["rolling_mean"] + 1.2 * df["rolling_std"])
).astype(int)

# BACKUP SAFETY
if df["outbreak"].mean() < 0.05:
    threshold = df["total_cases"].quantile(0.90)
    df["outbreak"] = (df["total_cases"] > threshold).astype(int)

# CLEAN
df = df.drop(columns=["noise"])

# SAVE
final_path = os.path.join(DATA_PROCESSED, "final_modeling_dataset_READY.csv")
df.to_csv(final_path, index=False)

print("\nPhase 5 COMPLETE")
print("Shape:", df.shape)
print("Outbreak rate:", round(df["outbreak"].mean(),3))

=== Phase 5: FINAL CLEAN DATASET ===

Raw shape: (24300, 20)

Phase 5 COMPLETE
Shape: (24300, 38)
Outbreak rate: 0.134
